# 03 · FunnyBirds + MCBM — does minimality fix grounding?  *(seed-aware)*

**Claim (MCBM):** IB makes `z_j` a minimal sufficient statistic of `c_j`. Loss
`L = CE + λ_c·BCE(z,c) + γ·0.2·mean((6c−3 − z)²)`. **Hypothesis:** minimality constrains
*content*, backwash is about *source*; when `c=f(class)` the ±3 target is class-derived,
so tightening γ can't remove class-reading. Decision rule + null criteria: `DECISIONS §D.5`.
All cells aggregate over seeds. *Refs: `fb_mcbm_renderer_swap.ipynb`, `fb_mcbm_rl_renderer_swap.ipynb`.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
EPS = 1e-3
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok
# ---- seed-aware grounding loaders ----
def load_grounding(prefix):
    """All seeds for a config prefix -> one df with a 'seed' column (None if absent)."""
    fs = sorted(glob.glob(str(CURATED/"grounding"/f"{prefix}-s*.parquet")))
    if not fs: return None
    out=[]
    for f in fs:
        d=pd.read_parquet(f); d["seed"]=int(re.search(r"-s(\d+)\.parquet$", f).group(1)); out.append(d)
    return pd.concat(out, ignore_index=True)
def per_part_seedagg(df, visible_only=True):
    """per (seed,part) retained_frac -> per-part mean/std/count across seeds."""
    d = df[df["changed_frac"]>EPS] if (visible_only and "changed_frac" in df.columns) else df
    g = d.groupby(["seed","part"]).agg(pi=("p_intact","mean"), pr=("p_removed","mean"))
    g["rf"] = g.pr/g.pi
    return g["rf"].groupby("part").agg(["mean","std","count"])


## Terms — read this first
- **minimality** = the *property* the MCBM enforces: each bottleneck slot `z_j` should carry
  *only* what predicts its own concept `c_j`, nothing else.
- **γ (gamma)** = the *knob* that controls how hard minimality is pushed — the coefficient on
  the minimality term in the loss; **effective force = γ×0.2**. γ=0 → off (≈ plain bottleneck);
  γ=5 → hard. So "vs γ" on every x-axis = "as we crank the minimality knob from off to hard."
- **z** = the raw per-concept bottleneck latent (26 slots, pre-concept-head).
  **c_preds** = concept probabilities (the readable layer, concept-head on `z`).
- **retained_frac** = P(concept | part deleted) / P(concept | intact). ~0 = grounded (concept
  vanished with its part); ~1 = **backwash** (still fires without its part).
- **margin** (swap test) = z[donor] − z[source] on the re-rendered image after a part is swapped
  to another species' variant. **>0** = the model noticed the new part (grounded); **<0** = it
  still reports the old part (**violation = backwash**).
- **ordering_correct** = (margin > 0). Averaged = the grounding accuracy shown in the heatmaps.
- **species←c_preds** = accuracy of guessing the *species* from the concept vector alone
  (chance = 1/50 = 0.02). ~0.99 ⇒ the concept layer is effectively a species code.
- **rep_loss / mean|z|** ("did γ bite") = how tightly `z` is pinned to the ±3 concept target;
  as γ rises rep_loss ↓ and mean|z| → 3 — that's how we confirm the knob actually moved `z`.

**Why some plots below look sparse.** The deletion + "did γ bite" sweeps ran all six γ, but the
**renderer-swap** job is the expensive one (live renderer + GPU, ~90 min/model), so only
**γ=0 and γ=5** swap CSVs exist yet → swap plots show 2 points / 2 heatmap rows. The **species
probe** was collected for **γ=3 only** so far → that plot shows 1 point. Both fill in once the
full sweeps run:
```
bash analysis/grounding_sweep.sh                         # all-γ deletion + species probe
CONFIG_PREFIX=funnybirds-mcbm GAMMAS="0 0.1 0.3 1 3 5" \
  SEEDS="1 2 3" sbatch train/renderer_swap.slurm         # full-γ swap (the money sweep)
```

## 1 · Overall `retained_frac` vs γ (±seed std) — the summary (diluted)
`backwash_vs_gamma.csv` (collected on **visible-only** rows). Averages all 5 parts, so
sits near 0.1 regardless — §2 is the one to show. Error bars = std across seeds.

In [ ]:
bw = CURATED/"backwash_vs_gamma.csv"
if need(bw, 'bash analysis/grounding_sweep.sh'):
    T = pd.read_csv(bw); display(T.round(3))
    mc = T[(T.model=="mcbm") & T.retained_frac.notna()]; cb = T[T.model=="cbm"]
    g = mc.groupby("gamma").retained_frac.agg(["mean","std"]).reset_index()
    floor=(g.gamma[g.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.errorbar(g.gamma.replace(0,floor), g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3, color=MCBM_C, label="MCBM")
    if len(cb): ax.axhline(cb.retained_frac.mean(), ls="--", color=CBM_C, label="CBM (ref)")
    ax.set_xscale("log"); ax.set_xlabel("γ (effective force = γ×0.2)"); ax.set_ylabel("overall retained_frac")
    ax.set_ylim(0,1.02); ax.set_title("Overall removed-part retention vs γ"); ax.legend()
    print("seeds per gamma:", mc.groupby("gamma").seed.nunique().to_dict())

📊 Overall retained_frac is **flat across γ (~0.09–0.11) and ≈ CBM** — but it is diluted by the 4 near-zero parts; §2 is the real view.

## 2 · Per-part `retained_frac` vs γ — **the figure**  (tail, ±seed band)
Read every seed's grounding parquet, visible-only, break out by part. Does the **tail**
curve come down as γ rises? (Shaded = ±std across seeds.)

In [ ]:
recs=[]
for f in sorted(glob.glob(str(CURATED/"grounding"/"funnybirds-*-s*.parquet"))):
    pr=parse_stem(Path(f).stem)
    if pr is None or pr[0]=="vanilla": continue
    model,gamma,seed=pr; d=pd.read_parquet(f)
    if "changed_frac" in d.columns: d=d[d["changed_frac"]>EPS]
    gg=d.groupby("part").agg(pi=("p_intact","mean"),pr=("p_removed","mean"))
    for part,row in gg.iterrows(): recs.append(dict(model=model,gamma=gamma,seed=seed,part=part,retained_frac=row.pr/row.pi))
P=pd.DataFrame(recs)
if len(P):
    parts=["tail","wing","beak","foot","eye"]
    A=P.groupby(["model","gamma","part"]).retained_frac.agg(["mean","std"]).reset_index()
    mc=A[A.model=="mcbm"]; cbm_pp=A[A.model=="cbm"].set_index("part")["mean"]
    floor=(mc.gamma[mc.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(7,4.4)); cmap=plt.cm.viridis(np.linspace(0,0.85,len(parts)))
    for part,col in zip(parts,cmap):
        s=mc[mc.part==part].sort_values("gamma");
        if not len(s): continue
        x=s.gamma.replace(0,floor)
        ax.plot(x,s["mean"],"o-",color=col,lw=3 if part=="tail" else 1.4,label=part+("  ← highest retention" if part=="tail" else ""))
        ax.fill_between(x, s["mean"]-s["std"].fillna(0), s["mean"]+s["std"].fillna(0), color=col, alpha=0.15)
        if part in cbm_pp.index: ax.scatter([floor/1.6],[cbm_pp[part]],marker="*",s=90,color=col,zorder=5)
    ax.set_xscale("log"); ax.set_xlabel("γ (effective force = γ×0.2)"); ax.set_ylabel("retained_frac (visible-only)")
    ax.set_ylim(-0.02,1.02); ax.set_title("Per-part retention vs γ (★=CBM)\nminimality does not bring tail down"); ax.legend(title="part",fontsize=8)
    print("tail retained_frac by γ (mean±std over seeds):")
    display(mc[mc.part=="tail"][["gamma","mean","std"]].sort_values("gamma").round(3))
else: print("[pending] bash analysis/grounding_sweep.sh")

📊 **Is the tail "coming down then rising" real? No — that's noise.** tail retained_frac by γ is
`0.37 · 0.38 · 0.42 · 0.36 · 0.27 · 0.45` — it wiggles in a 0.27–0.45 band with **no monotone
trend**, on a **single seed**. The γ=3 dip and γ=5 jump are within run-to-run noise; don't read a
"minimality helps then breaks" story into 6 unreplicated points. Honest read: tail retention is
**flat and high** across γ while the other 4 parts sit near 0 — minimality doesn't bring tail down.
(≥3 seeds + CI before claiming any finer shape — `DECISIONS §D.5`.)

## 3 · Species-code vs γ (seed-averaged) — did the class channel survive?
If `species←c_preds` stays ≈1 as γ grows, minimality compressed the representation
without cutting the class channel.

In [ ]:
rows=[]
for f in sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-mcbm-g*-s*.json"))):
    m=re.search(r"-g([0-9p]+)-s(\d+)\.json$", Path(f).name)
    if not m: continue
    S=json.loads(Path(f).read_text())
    rows.append(dict(gamma=float(m.group(1).replace("p",".")), seed=int(m.group(2)),
                     c=S["species_from_cpreds"]["acc"], tail=S["species_from_part_cpreds"].get("tail",{}).get("acc",np.nan),
                     chance=S["chance"]))
if rows:
    D=pd.DataFrame(rows); Dg=D.groupby("gamma").agg(c=("c","mean"),tail=("tail","mean"),chance=("chance","first")).reset_index()
    display(Dg.round(3)); floor=(Dg.gamma[Dg.gamma>0].min() or 0.05)/3
    cbj=sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-cbm-s*.json")))
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.plot(Dg.gamma.replace(0,floor),Dg.c,"o-",color=MCBM_C,label="species←c_preds")
    ax.plot(Dg.gamma.replace(0,floor),Dg["tail"],"s--",color="#5B8C5A",label="species←tail concepts")
    if cbj: ax.axhline(np.mean([json.loads(Path(p).read_text())["species_from_cpreds"]["acc"] for p in cbj]),ls=":",color=CBM_C,label="CBM species←c_preds")
    ax.axhline(Dg.chance.iloc[0],ls=":",color="k",label="chance"); ax.set_xscale("log")
    ax.set_xlabel("γ"); ax.set_ylabel("species recoverable"); ax.set_ylim(0,1.02); ax.legend(fontsize=8)
    ax.set_title("Class channel survives minimality")
else: print("[pending] grounding_sweep.sh runs the probe too")

📊 **species←c_preds ≈ 0.99, tail concepts ≈ 0.27** (chance 0.02) → the concept layer encodes species. **Only γ=3 has a probe collected so far** (hence the single point) — run `analysis/grounding_sweep.sh` to fill the other five γ and confirm the code survives across the whole minimality range.

## 4 · CONTROL — did γ actually tighten the bottleneck? (seed-averaged)
Flat retention only refutes minimality **if γ changed the representation**. Read `z` from
saved predictions: `mean((6c−3 − z)²)` ↓ / `mean|z|` ↑ with γ = γ bit. See `DECISIONS §D.5`.

In [ ]:
import torch
def zstats_seed(cfg, seed):
    pth = REPO/"external"/"minimal_cbm"/"results"/cfg/str(seed)/"predictions"/"epoch_100.pth"
    if not pth.exists(): return None
    d=torch.load(pth, map_location="cpu", weights_only=False); z,cc=d["z"].float(),d["c"].float()
    if not (np.isfinite(z).all() and np.isfinite(cc).all()): return None
    yp=d["y_preds"]; ta=float((yp.argmax(-1)==d["y"]).float().mean()) if yp.ndim>1 else float((yp==d["y"]).float().mean())
    cp=d["c_preds"]; cp=cp[...,0] if cp.ndim==3 else cp
    return float(((6*cc-3-z)**2).mean()), float(z.abs().mean()), ta, float(((cp>=0.5).float()==cc).float().mean())
rows=[]
for g,tag in [(0,"g0"),(0.1,"g0p1"),(0.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    for seed in range(1,6):
        s=zstats_seed(f"funnybirds-mcbm-{tag}", seed)
        if s: rows.append((g,seed,*s))
if rows:
    D=pd.DataFrame(rows,columns=["gamma","seed","rep_loss","mean_abs_z","task_acc","concept_acc"])
    Dg=D.groupby("gamma").agg(rep_loss=("rep_loss","mean"),mean_abs_z=("mean_abs_z","mean"),
                              task_acc=("task_acc","mean"),concept_acc=("concept_acc","mean"),n_seeds=("seed","nunique")).reset_index()
    display(Dg.round(3)); floor=(Dg.gamma[Dg.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(1,2,figsize=(11,3.8))
    ax[0].plot(Dg.gamma.replace(0,floor),Dg.rep_loss,"o-",color=MCBM_C,label="mean (±3−z)²")
    a0=ax[0].twinx(); a0.plot(Dg.gamma.replace(0,floor),Dg.mean_abs_z,"s--",color="#5B8C5A",label="mean|z|")
    ax[0].set_xscale("log"); ax[0].set_xlabel("γ"); ax[0].set_ylabel("minimality term (↓=pinned)"); a0.set_ylabel("mean|z|"); ax[0].set_title("Did γ bite?")
    ax[1].plot(Dg.gamma.replace(0,floor),Dg.task_acc,"o-",label="task"); ax[1].plot(Dg.gamma.replace(0,floor),Dg.concept_acc,"s--",label="concept")
    ax[1].set_xscale("log"); ax[1].set_xlabel("γ"); ax[1].set_ylabel("val acc"); ax[1].set_ylim(0,1.02); ax[1].legend(); ax[1].set_title("Fit vs γ")
    plt.tight_layout()
    moved=(Dg.rep_loss.max()-Dg.rep_loss.min())>0.05*max(Dg.rep_loss.max(),1e-9) or (Dg.mean_abs_z.max()-Dg.mean_abs_z.min())>0.1
    print("VERDICT:", "γ moved the representation -> flat retention is a real refutation" if moved
          else "γ barely moved -> sweep underpowered; WIDEN γ (DECISIONS §D.5)")
else: print("[pending] need results/funnybirds-mcbm-g*/<seed>/predictions/epoch_100.pth")

📊 **γ genuinely bit** — the minimality term drops (rep_loss 458→~0.15, mean|z|→±3). So the flat backwash above is a real refutation, not an inert sweep (see DECISIONS §D.5).

## 5 · IB compression vs grounding — *your §20, deletion-adapted*
*(ports `fb_mcbm_renderer_swap.ipynb §20` "IB compression vs visual swap response")*.
The original plotted z-scale (compression) vs the part-**swap** response; we don't have
the live renderer in curated yet, so we use the **deletion** signal we do have: does IB
compressing `z` (z-std ↓, `mean|z|` up toward ±3) coincide with the tail becoming
**grounded** (retained_frac ↓)? If z compresses but retention stays flat, **compression
is not selective for grounding** — the §20 conclusion, on curated models.

In [ ]:
import torch
def zscale(cfg):
    vals=[]
    for seed in range(1,6):
        p=REPO/"external"/"minimal_cbm"/"results"/cfg/str(seed)/"predictions"/"epoch_100.pth"
        if not p.exists(): continue
        d=torch.load(p,map_location="cpu",weights_only=False); z,cc=d["z"].float(),d["c"].float()
        if not np.isfinite(z).all(): continue
        vals.append((float(z.std()), float(z.abs().mean())))
    return np.array(vals).mean(0) if vals else None
def tail_ret(tag):
    rf=[]
    for f in glob.glob(str(CURATED/"grounding"/f"funnybirds-mcbm-{tag}-s*.parquet")):
        d=pd.read_parquet(f); d=d[d.part=="tail"]
        if "changed_frac" in d.columns: d=d[d["changed_frac"]>EPS]
        if len(d) and d.p_intact.mean()>1e-6: rf.append(d.p_removed.mean()/d.p_intact.mean())
    return np.mean(rf) if rf else np.nan
rows=[]
for g,tag in [(0,"g0"),(0.1,"g0p1"),(0.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    zs=zscale(f"funnybirds-mcbm-{tag}"); tr=tail_ret(tag)
    if zs is not None and np.isfinite(tr): rows.append((g,zs[0],zs[1],tr))
if rows:
    D=pd.DataFrame(rows,columns=["gamma","z_std","mean_abs_z","tail_retained"]); display(D.round(3))
    floor=(D.gamma[D.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(D.gamma.replace(0,floor),D.z_std,"o-",color="steelblue",label="z std (spread)")
    a=ax[0].twinx(); a.plot(D.gamma.replace(0,floor),D.mean_abs_z,"s--",color="crimson",label="mean|z| (toward +-3)")
    ax[0].set_xscale("log"); ax[0].set_xlabel("gamma"); ax[0].set_ylabel("z std"); a.set_ylabel("mean|z|")
    ax[0].set_title("IB compresses z as gamma increases"); ax[0].legend(loc="upper left",fontsize=8); a.legend(loc="upper right",fontsize=8)
    sc=ax[1].scatter(D.z_std,D.tail_retained,c=D.gamma,cmap="viridis",s=90)
    for _,r in D.iterrows(): ax[1].annotate(f"g={r.gamma:g}",(r.z_std,r.tail_retained),fontsize=8)
    ax[1].set_xlabel("z std  (more compression <-)"); ax[1].set_ylabel("tail retained_frac")
    ax[1].set_title("Compression does NOT buy grounding (retention ~flat)"); plt.colorbar(sc,ax=ax[1],label="gamma")
    plt.tight_layout()
    print("z compresses with gamma but tail retained_frac stays ~flat -> IB compression is not selective for part grounding (section 20).")
else:
    print("[pending] need mcbm predictions (epoch_100.pth) + grounding parquets")

📊 z compresses as γ rises, but **tail retention stays flat** → IB compression is not selective for part grounding (your §20).

## 6 · Renderer-swap z-ordering vs γ — does minimality fix grounding?  *(ref §13–19)*
Same swap as CBM (notebook 02 §6), swept over the minimality strength γ. The question:
as the information bottleneck tightens, does the **tail** start correctly detecting the
swapped-in part (grounding restored), or does it stay backwashed? Reads the MCBM swap
CSVs `swap/funnybirds-mcbm-g*-s1.csv` (produced by
`CONFIG_PREFIX=funnybirds-mcbm sbatch train/renderer_swap.slurm`) + the CBM CSV as reference.

In [ ]:
ORDER=["tail","wing","beak","foot","eye"]
def load_swaps():
    rows=[]
    for fp in sorted(glob.glob(str(CURATED/"swap"/"funnybirds-mcbm-g*-s1.csv"))):
        m=re.search(r"-g([0-9p]+)-s",Path(fp).name)
        if not m: continue
        d=pd.read_csv(fp); d["gamma"]=float(m.group(1).replace("p",".")); rows.append(d)
    SW=pd.concat(rows,ignore_index=True) if rows else None
    cb=CURATED/"swap"/"funnybirds-cbm-s1.csv"
    return SW, (pd.read_csv(cb) if cb.exists() else None)
SW,CB=load_swaps()
if SW is None:
    print("[pending] no MCBM swap CSVs -> CONFIG_PREFIX=funnybirds-mcbm GAMMAS=\"0 0.1 0.3 1 3 5\" sbatch train/renderer_swap.slurm")
else:
    H=SW.groupby(["gamma","part"]).ordering_correct.mean().unstack().reindex(columns=ORDER)
    display(H.round(3))
    fig,ax=plt.subplots(figsize=(6.2,3.8)); im=ax.imshow(H.values,cmap="RdYlGn",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER)
    ax.set_yticks(range(len(H.index))); ax.set_yticklabels([f"γ={g:g}" for g in H.index])
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            v=H.values[i,j]
            if np.isfinite(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=8)
    ax.set_title("z-ordering_correct — part × γ (green=grounded, red=backwash)"); fig.colorbar(im,fraction=0.046)

📊 **How to read this grid (it's not a confusion matrix — it's part × γ).** Each **row** is a
minimality setting (only γ=0 and γ=5 have swap data so far); each **column** is a body part; each
**cell** = `ordering_correct` = the fraction of swaps where, after that part was swapped to a donor
variant, the **donor's** concept correctly out-fired the original part's concept. **Green ≈ 1 =
grounded** (the model saw the new part); **red ≈ 0 = backwash** (it still reports the old part =
reading species, not pixels). Read the **tail** column: 0.36 at γ=0 → 0.20 at γ=5 — worst part, and
it gets slightly *worse* with minimality. foot 0.86→0.95 stays grounded. So tightening minimality
does **not** restore tail grounding.

In [ ]:
if SW is not None:
    t=SW[SW.part=="tail"].groupby("gamma").ordering_correct.mean()
    floor=(t.index[t.index>0].min() or 0.05)/3 if (t.index>0).any() else 0.02
    fig,ax=plt.subplots(figsize=(6,3.6))
    ax.plot([g if g>0 else floor for g in t.index], t.values,"o-",color=MCBM_C,label="MCBM tail")
    if CB is not None:
        ax.axhline(CB[CB.part=="tail"].ordering_correct.mean(),ls="--",color=CBM_C,label="CBM tail (ref)")
    ax.axhline(0.5,ls=":",color="gray"); ax.set_xscale("log"); ax.set_ylim(0,1.02)
    ax.set_xlabel("γ (minimality knob = γ×0.2)"); ax.set_ylabel("tail ordering_correct"); ax.legend()
    ax.set_title("Does minimality fix the tail swap?")
else: print("[pending] mcbm swap CSVs")

📊 **Tail grounding vs γ — and yes, it goes *down* (0.36→0.19).** Higher minimality makes the tail *slightly worse*, not better — the opposite of "minimality fixes grounding." It sits at/below the CBM dashed line and below the 0.5 coin-flip the whole way. **Caveat:** only 2 γ points (0 and 5), single seed — read this as *directional* ("not fixed, if anything worse"), not a precise slope. The full γ sweep + seeds firm it up.

### E · Exploratory — *does the tail margin climb as minimality tightens?*  *(dots + distribution)*
The raw per-swap **dots are back** — but coloured by **swap direction** (fwd/bwd), *not* by the
margin's own sign (that would just repaint the two sides of the line and tell you nothing). On top
we draw the **box** (the honest distribution, whiskers 5–95%) and **foot's median as the grounded
reference**. If minimality helped, the tail dots/box would drift **up off the red line** toward foot
as γ grows; if they sit on 0 at every γ, minimality does nothing for the tail.

In [ ]:
if SW is not None:
    gammas=sorted(SW.gamma.unique()); rs=np.random.RandomState(0)
    tdata=[SW[(SW.part=="tail")&(SW.gamma==g)].margin.values for g in gammas]
    has_dir="direction" in SW.columns
    fig,ax=plt.subplots(figsize=(7.8,4.2))
    # raw dots (you asked) — coloured by DIRECTION (independent of the margin's sign), not by viol
    for i,g in enumerate(gammas):
        d=SW[(SW.part=="tail")&(SW.gamma==g)]
        if has_dir:
            for dirn,col in [("fwd","#1f77b4"),("bwd","#d62728")]:
                dd=d[d.direction==dirn]; xj=rs.normal(i+1,0.08,len(dd))
                ax.scatter(xj,dd.margin,s=7,alpha=0.3,color=col,label=(dirn if i==0 else None))
        else:
            xj=rs.normal(i+1,0.08,len(d)); ax.scatter(xj,d.margin,s=7,alpha=0.3,color="#6a0dad")
    # box overlay = the honest distribution summary (hollow so dots show through)
    bp=ax.boxplot(tdata,showfliers=False,patch_artist=True,whis=(5,95),medianprops=dict(color="k"),widths=0.5)
    for patch in bp["boxes"]: patch.set_facecolor("none"); patch.set_edgecolor("k")
    footmed=[np.median(SW[(SW.part=="foot")&(SW.gamma==g)].margin) if ((SW.part=="foot")&(SW.gamma==g)).any() else np.nan for g in gammas]
    ax.plot(range(1,len(gammas)+1),footmed,"s--",color="#2ca02c",lw=2,label="foot median (grounded ref)")
    ax.axhline(0,color="r",lw=1,ls="--",label="ordering boundary (0)")
    ax.set_xticks(range(1,len(gammas)+1)); ax.set_xticklabels([f"γ={g:g}" for g in gammas])
    ax.set_xlabel("γ (minimality knob = γ×0.2)"); ax.set_ylabel("tail margin after swap (donor − source)")
    ax.legend(fontsize=8); ax.set_title("Tail margin per γ — raw swaps (dots, by direction) + box; does it climb toward foot?")
else: print("[pending] mcbm swap CSVs")

📊 If the tail box stays pinned on the red line (median ≈ 0) across γ while foot's reference sits well above it, minimality is not moving the tail toward grounding — the causal fix, if any, has to come from the *labels* (notebook 03rl), not the bottleneck strength.

In [ ]:
if SW is not None and "pixel_count_cf" in SW.columns:
    allg=SW[SW.part=="tail"].groupby("gamma").ordering_correct.mean()
    visg=SW[(SW.part=="tail")&(SW.pixel_count_cf>=50)].groupby("gamma").ordering_correct.mean()
    footv=SW[(SW.part=="foot")&(SW.pixel_count_cf>=50)].groupby("gamma").ordering_correct.mean()
    D=pd.DataFrame({"tail_all":allg,"tail_visible":visg,"foot_visible(grounded)":footv}); display(D.round(3))
    fig,ax=plt.subplots(figsize=(6.4,3.6)); x=np.arange(len(D))
    ax.plot(x,D["tail_all"],"o-",label="tail all swaps",color="#9ec9e2")
    ax.plot(x,D["tail_visible"],"s-",label="tail visible-only",color=MCBM_C)
    ax.plot(x,D["foot_visible(grounded)"],"^--",label="foot visible-only (grounded ref)",color="#2ca02c")
    ax.set_xticks(x); ax.set_xticklabels([f"γ={g:g}" for g in D.index]); ax.set_ylim(0,1.02)
    ax.set_ylabel("ordering_correct"); ax.legend(fontsize=7); ax.set_title("Occlusion control, per γ — tail vs grounded foot")
else: print("[pending] mcbm swap CSVs with pixel_count_cf")

📊 **Occlusion control across γ.** visible-only stays well below grounded at every γ → tail backwash survives both the visibility filter *and* minimality.

### The training-occlusion confound also applies here — and γ can't remove it
The occlusion control above only manipulates **test-time** pixels. The deeper confound is the
same one flagged in notebook 02 §E: the tail is the most-often-occluded part *during training*,
so the encoder may simply be **under-trained** on it — a low tail margin can mean "species-anchored
(backwash)" **or** "rarely got a clean gradient." Crucially, **tightening γ cannot separate these
two**: minimality reshapes how `z` encodes a concept, but it does not change *which pixels were
visible during training* nor the `concept = f(species)` label correlation. That is exactly why the
flat tail curve across γ, on its own, does **not** settle the question — it only rules out "the
bottleneck was too loose." The clean disentangler is **notebook 03rl**, where the concept↔species
correlation is broken by relabeling while visibility stays fixed: if tail grounding recovers there,
the labels (not the bottleneck) were the lever.

### 6d · z-ordering per part vs γ — all parts  *(ref §15)*
The full cross-γ view: every part's swap-detection accuracy as minimality tightens.

In [ ]:
if SW is not None:
    fig,ax=plt.subplots(figsize=(6.6,4))
    for part,col in zip(ORDER, plt.cm.tab10.colors):
        s=SW[SW.part==part].groupby("gamma").ordering_correct.mean()
        ax.plot([g if g>0 else 0.02 for g in s.index], s.values,"o-",color=col,lw=3 if part=="tail" else 1.4,label=part)
    ax.set_xscale("log"); ax.axhline(0.5,ls=":",color="gray"); ax.axhline(1.0,ls=":",color="green"); ax.set_ylim(0,1.05)
    ax.set_xlabel("γ (minimality knob = γ×0.2)"); ax.set_ylabel("ordering_correct"); ax.legend(fontsize=8)
    ax.set_title("z-ordering per part vs γ")
else: print("[pending] mcbm swap CSVs")

📊 tail stays lowest at every γ; foot/wing stay grounded → minimality doesn't reshuffle which parts are backwashed.

### 6e · CBM vs MCBM(γ) — combined grounding heatmap  *(ref §22)*
The money comparison on one grid: CBM as the top row, then MCBM at each γ. Does *any*
row (any γ) turn the tail column green? If not, neither the bottleneck nor its strength
fixes tail grounding.

In [ ]:
if SW is not None:
    rows=[]
    if CB is not None: rows.append(("CBM", CB.groupby("part").ordering_correct.mean()))
    for g,d in SW.groupby("gamma"): rows.append((f"MCBM γ={g:g}", d.groupby("part").ordering_correct.mean()))
    Hc=pd.DataFrame({n:s for n,s in rows}).T.reindex(columns=ORDER)
    display(Hc.round(3))
    fig,ax=plt.subplots(figsize=(6.2,0.5*len(Hc)+1.5)); im=ax.imshow(Hc.values,cmap="RdYlGn",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER); ax.set_yticks(range(len(Hc))); ax.set_yticklabels(Hc.index)
    for i in range(Hc.shape[0]):
        for j in range(Hc.shape[1]):
            v=Hc.values[i,j]
            if np.isfinite(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=8)
    ax.set_title("Grounding: CBM vs MCBM(γ)  (green=grounded)"); fig.colorbar(im,fraction=0.046)
else: print("[pending] mcbm swap CSVs")

📊 **The headline comparison (ref §22).** The tail column stays red across CBM and every MCBM γ → the bottleneck (and tightening it) does not fix tail backwash.

## 7 · Per-γ detail figures — the reference "detail" cluster, ported  *(ref §14, §43–70)*
These are the individual-swap views from the reference MCBM notebook (the red/blue dot plots,
per-species and per-concept breakdowns, z-distribution), now on the curated swap CSVs. They read
whatever γ are present (currently γ=0,5) and pick the first available γ for the single-γ details.
Where the reference coloured dots by the margin's sign, we colour by an **independent** variable
(direction / visibility) so the colour carries information.

In [ ]:
if SW is not None:
    fig,ax=plt.subplots(figsize=(6.8,4))
    for part,col in zip(ORDER, plt.cm.tab10.colors):
        s=SW[SW.part==part].groupby("gamma").margin.mean()
        ax.plot([g if g>0 else 0.02 for g in s.index], s.values,"o-",color=col,lw=3 if part=="tail" else 1.4,label=part)
    ax.axhline(0,color="r",ls="--",lw=1); ax.set_xscale("log"); ax.set_xlabel("γ (minimality knob = γ×0.2)")
    ax.set_ylabel("mean z-ordering margin (donor − source)"); ax.legend(fontsize=8)
    ax.set_title("Mean z-ordering margin per part vs γ  (ref §14)")
else: print("[pending] mcbm swap CSVs")

📊 Higher = the bottleneck more decisively detects the swapped-in part. tail hugs/sits below 0 at every γ; foot/wing stay well above → minimality doesn't lift the tail margin.

In [ ]:
if SW is not None:
    Hv=(1-SW.groupby(["gamma","part"]).ordering_correct.mean().unstack()).reindex(columns=ORDER)
    display(Hv.round(3))
    fig,ax=plt.subplots(figsize=(6.2,0.6*len(Hv)+1.5)); im=ax.imshow(Hv.values,cmap="Reds",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER)
    ax.set_yticks(range(len(Hv.index))); ax.set_yticklabels([f"γ={g:g}" for g in Hv.index])
    for i in range(Hv.shape[0]):
        for j in range(Hv.shape[1]):
            v=Hv.values[i,j]
            if np.isfinite(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=8)
    ax.set_title("Violation rate (1 − ordering_correct) — part × γ  (ref §15)"); fig.colorbar(im,fraction=0.046)
else: print("[pending] mcbm swap CSVs")

📊 The complement of the grounding heatmap — dark red = backwash. tail is the darkest column at every γ.

In [ ]:
if SW is not None:
    gammas=sorted(SW.gamma.unique()); rs=np.random.RandomState(1); has_dir="direction" in SW.columns
    fig,ax=plt.subplots(figsize=(9,4.2)); xt=[]; xl=[]; pos=0
    for g in gammas:
        for part in ORDER:
            dd0=SW[(SW.gamma==g)&(SW.part==part)]
            if has_dir:
                for dirn,col in [("fwd","#1f77b4"),("bwd","#d62728")]:
                    e=dd0[dd0.direction==dirn]; ax.scatter(rs.normal(pos,0.06,len(e)),e.margin,s=5,alpha=0.25,color=col,label=(dirn if (pos==0) else None))
            else:
                ax.scatter(rs.normal(pos,0.06,len(dd0)),dd0.margin,s=5,alpha=0.25,color="#6a0dad")
            xt.append(pos); xl.append(f"{part}\nγ{g:g}"); pos+=1
        pos+=0.6
    ax.axhline(0,color="k",lw=0.8); ax.set_xticks(xt); ax.set_xticklabels(xl,fontsize=6,rotation=90)
    ax.set_ylabel("z-ordering margin"); ax.legend(fontsize=8)
    ax.set_title("Margin — all γ × parts (dots by direction; below 0 = violation)")
else: print("[pending] mcbm swap CSVs")

📊 The full raw view: every part at every γ. tail columns sit low and spread across 0 at both γ; foot/wing columns ride above 0 — the split doesn't change as minimality tightens.

In [ ]:
if SW is not None and {"z_old_orig","z_new_orig"}.issubset(SW.columns):
    g0=sorted(SW.gamma.unique())[0]; s=SW[SW.gamma==g0]
    fig,ax=plt.subplots(figsize=(5.4,5))
    for part,col in zip(ORDER, plt.cm.tab10.colors):
        e=s[s.part==part]; ax.scatter(e.z_old_orig,e.z_new_orig,s=6,alpha=0.3,color=col,label=part)
    lo=float(s[["z_old_orig","z_new_orig"]].min().min()); hi=float(s[["z_old_orig","z_new_orig"]].max().max())
    ax.plot([lo,hi],[lo,hi],"k--",lw=0.5); ax.set_xlabel("z source concept (part present)"); ax.set_ylabel("z donor concept (part ABSENT)")
    ax.legend(fontsize=7); ax.set_title(f"Grounding before swap  γ={g0:g}  (high y = backwash)")
else: print("[pending] swap CSV with z_old_orig/z_new_orig")

📊 Donor concept hugs its floor on original images (points along the x-axis) → on-distribution grounding, same as CBM §5; backwash only appears once the part is swapped in.

In [ ]:
if SW is not None and "p_cf_donor" in SW.columns:
    g0=sorted(SW.gamma.unique())[0]; t=SW[(SW.part=="tail")&(SW.gamma==g0)].copy()
    fig,ax=plt.subplots(figsize=(6.2,4))
    if "pixel_count_cf" in t.columns:
        sc=ax.scatter(t.margin,t.p_cf_donor,s=7,alpha=0.3,c=np.log10(t.pixel_count_cf.clip(lower=1)),cmap="viridis"); fig.colorbar(sc,label="log10 tail px")
    else:
        ax.scatter(t.margin,t.p_cf_donor,s=7,alpha=0.3,color="#888")
    if len(t)>8:
        t["mb"]=pd.qcut(t.margin,8,duplicates="drop"); gb=t.groupby("mb",observed=True).agg(mx=("margin","mean"),my=("p_cf_donor","mean"))
        ax.plot(gb.mx,gb.my,"o-",color="crimson",lw=2,label="binned mean"); ax.legend(fontsize=8)
    ax.axvline(0,color="k",lw=0.5); ax.set_xlabel("tail z-ordering margin"); ax.set_ylabel("P(donor species) after swap")
    ax.set_title(f"Downstream: does margin move species prob?  γ={g0:g}")
else: print("[pending] swap CSV with p_cf_donor")

📊 Coloured by visibility (independent of the margin's sign); the crimson **binned mean** is the honest trend — a larger margin nudges P(donor species) up → the failure is upstream in the concept layer, not the classifier.

In [ ]:
if SW is not None and "sid_src" in SW.columns:
    g0=sorted(SW.gamma.unique())[0]; d0=SW[(SW.part=="tail")&(SW.gamma==g0)]
    d0=d0.assign(viol=~d0.ordering_correct.astype(bool)); sv=d0.groupby("sid_src").viol.mean().sort_values(ascending=False)
    fig,ax=plt.subplots(figsize=(9,3)); ax.bar(sv.index.astype(str),sv.values,color=MCBM_C)
    ax.set_xlabel("source species"); ax.set_ylabel("tail violation rate"); ax.set_title(f"Per-source-species tail violation  γ={g0:g}")
    plt.xticks(rotation=90,fontsize=6)
else: print("[pending] swap CSV with sid_src")

📊 Some source species' tails are almost never detected after a swap → species-specific backwash, not uniform off-distribution confusion (mirrors CBM §3c).

In [ ]:
if SW is not None and "var_donor" in SW.columns:
    g0=sorted(SW.gamma.unique())[0]; s=SW[SW.gamma==g0]
    vc=s.assign(viol=~s.ordering_correct.astype(bool)).groupby(["part","var_donor"]).viol.mean().sort_values(ascending=False).head(20)
    fig,ax=plt.subplots(figsize=(8,3.4)); ax.bar([f"{p}_{int(v)}" for p,v in vc.index],vc.values,color=MCBM_C)
    ax.set_ylabel("violation rate"); ax.set_title(f"Top-20 concept slots by violation  γ={g0:g}"); plt.xticks(rotation=90,fontsize=7)
else: print("[pending] swap CSV with var_donor")

📊 The worst-grounded slots are **tail variants** — they dominate the top of the violation list at MCBM just as at CBM.

In [ ]:
if SW is not None and "pixel_count_cf" in SW.columns:
    g0=sorted(SW.gamma.unique())[0]; t=SW[(SW.part=="tail")&(SW.gamma==g0)]
    fig,ax=plt.subplots(1,2,figsize=(11,3.4))
    ax[0].scatter(t.pixel_count_cf,t.margin,s=6,alpha=0.3,color=MCBM_C); ax[0].axhline(0,color="k",lw=0.5); ax[0].axvline(50,ls=":",color="crimson")
    ax[0].set_xlabel("swapped-in tail pixels"); ax[0].set_ylabel("margin"); ax[0].set_title(f"Tail: visibility vs margin  γ={g0:g}")
    ok=t.ordering_correct.astype(bool)
    ax[1].hist(t.pixel_count_cf[ok],bins=30,alpha=0.5,color="#2ca02c",label="correct")
    ax[1].hist(t.pixel_count_cf[~ok],bins=30,alpha=0.5,color="#d62728",label="violation")
    ax[1].set_xlabel("swapped-in tail pixels"); ax[1].set_ylabel("count"); ax[1].legend(); ax[1].set_title("Tail visibility by outcome")
    plt.tight_layout()
else: print("[pending] swap CSV with pixel_count_cf")

📊 Violations occur across the whole visibility range, not only at low pixels → test-time occlusion doesn't explain the tail failures (matches the occlusion control above). Training-time occlusion is the confound 03rl addresses.

In [ ]:
import torch
def _load_z(cfg,seed=1):
    p=REPO/"external"/"minimal_cbm"/"results"/cfg/str(seed)/"predictions"/"epoch_100.pth"
    if not p.exists(): return None
    d=torch.load(p,map_location="cpu",weights_only=False); z=d["z"].float().numpy().ravel()
    z=z[np.isfinite(z)]; return z if len(z) else None
zm=_load_z("funnybirds-mcbm-g0"); zc=_load_z("funnybirds-cbm")
if zm is not None and zc is not None:
    fig,ax=plt.subplots(figsize=(6.4,3.8))
    ax.hist(zc,bins=80,alpha=0.5,color=CBM_C,density=True,label="CBM z")
    ax.hist(zm,bins=80,alpha=0.5,color=MCBM_C,density=True,label="MCBM γ=0 z")
    ax.set_xlabel("z (bottleneck latent)"); ax.set_ylabel("density"); ax.legend()
    ax.set_title("z distribution: MCBM γ=0 vs CBM  (ref §23 — why γ=0 ≠ CBM)")
else: print("[pending] need mcbm-g0 and cbm epoch_100.pth predictions")

📊 Even at γ=0 the MCBM `z` is **not** the CBM sigmoid-scale `z` — the heads/scales differ, so γ=0 is not "vanilla CBM" (documented in `DECISIONS`, "γ=0≠CBM"). This is why we keep a separate CBM notebook rather than treating MCBM γ=0 as the CBM baseline.

## Takeaway
If retention is flat/rising in γ while §4 shows γ tightened the rep and §3 shows species
stays recoverable, minimality shaped *content* but left the *class channel* intact.
**Lock only with ≥3 seeds + a bounded-CI (equivalence) statement — see `DECISIONS §D.5`.**

## How `retained_frac` is read as a backwash measurement
No axis is labelled "backwash"; the computed number is
**`retained_frac = P(concept | part removed) / P(concept | intact)`**, on **visible-only**
removals (the part actually left the render). Grounded → collapses to ~0; backwashed →
stays ~1. `retained_frac` is the metric; "concept–class backwash" is the interpretation.